# 第9章: 事前学習済み言語モデル（BERT型）

本章では、BERT型の事前学習済みモデルを利用して、マスク単語の予測や文ベクトルの計算、評判分析器（ポジネガ分類器）の構築に取り組む。

In [2]:
!pip install transformers
!pip install fugashi unidic-lite
!pip install evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 17.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.9/697.9 kB 2.6 MB/s eta 0:00:00
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658817 sha256=29d8ffe95a53f79b3737cd56bd1ea9934cb1630352f956696cb6b81c22856a57
  Stored in directory: /root/.cache/pip/wheels/5e/1f/0f/4d43887e5476d956fae828ee9b6687becd5544d68b51ed633d
Successfully built unidic-lite
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [3]:
from transformers import AutoTokenizer, AutoModel, pipeline, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from datasets import Dataset
import evaluate

In [4]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


## 80. トークン化

"The movie was full of incomprehensibilities."という文をトークンに分解し、トークン列を表示せよ。

In [5]:
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')
model = AutoModel.from_pretrained('google-bert/bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [5]:
text = "The movie was full of incomprehensibilities."
inputs = tokenizer(text, return_tensors = 'pt')
inputs

{'input_ids': tensor([[  101,  1996,  3185,  2001,  2440,  1997,  4297, 25377,  2890, 10222,
          5332, 14680,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

## 81. マスクの予測

"The movie was full of [MASK]."の"[MASK]"を埋めるのに最も適切なトークンを求めよ。

In [16]:
pipe = pipeline("fill-mask", model = "google-bert/bert-base-uncased")

text = "The movie was full of [MASK]."
outputs = pipe(text, top_k = 1)
print(f"input = {text}:")
print("outputs:")
for output in outputs:
  print(f"- {output}")

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


input = The movie was full of [MASK].:
outputs:
- {'score': 0.10711909830570221, 'token': 4569, 'token_str': 'fun', 'sequence': 'the movie was full of fun.'}


## 82. マスクのtop-k予測

"The movie was full of [MASK]."の"[MASK]"に埋めるのに適切なトークン上位10個と、その確率（尤度）を求めよ。

In [7]:
pipe = pipeline("fill-mask", model = "google-bert/bert-base-uncased")

text = "The movie was full of [MASK]."
outputs = pipe(text, top_k = 10)
print(f"input = {text}:")
print("outputs:")
for output in outputs:
  print(f"- {output}")

Device set to use cpu


input = The movie was full of [MASK].:
outputs:
- {'score': 0.431761234998703, 'token': 23418, 'token_str': 'you', 'sequence': 'The movie was full of you .'}
- {'score': 0.23234067857265472, 'token': 24294, 'token_str': 'me', 'sequence': 'The movie was full of me .'}
- {'score': 0.09867756813764572, 'token': 21801, 'token_str': 'it', 'sequence': 'The movie was full of it .'}
- {'score': 0.046985987573862076, 'token': 32498, 'token_str': 'music', 'sequence': 'The movie was full of music .'}
- {'score': 0.04039139300584793, 'token': 28478, 'token_str': 'man', 'sequence': 'The movie was full of man .'}
- {'score': 0.02070973999798298, 'token': 28187, 'token_str': 'all', 'sequence': 'The movie was full of all .'}
- {'score': 0.01607242412865162, 'token': 29, 'token_str': '.', 'sequence': 'The movie was full of . .'}
- {'score': 0.01055933814495802, 'token': 100, 'token_str': 'u', 'sequence': 'The movie was full of u .'}
- {'score': 0.005990787409245968, 'token': 13891, 'token_str': 'the', 

## 83. CLSトークンによる文ベクトル

以下の文の全ての組み合わせに対して、最終層の[CLS]トークンの埋め込みベクトルを用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


In [8]:
texts = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

cls_vectors = []
for text in texts:
  inputs = tokenizer(text, return_tensors = 'pt', padding = True, truncation = True)
  with torch.no_grad():
    outputs = model(**inputs)
  last_hidden_state = outputs.last_hidden_state # [バッチサイズ, シーケンス長, 隠れ層の次元数]
  cls_vector = last_hidden_state[0, 0, :] # [CLS]トークンのベクトル
  cls_vectors.append(cls_vector)

print("文の組み合わせに対するコサイン類似度: ")
similarity_matrix = cosine_similarity(cls_vectors)
for i in range(len(texts)):
  for j in range(i + 1, len(texts)):
    print(f"'{texts[i]}' と '{texts[j]}' の類似度: {similarity_matrix[i][j]:.4f}")

文の組み合わせに対するコサイン類似度: 
'The movie was full of fun.' と 'The movie was full of excitement.' の類似度: 0.9881
'The movie was full of fun.' と 'The movie was full of crap.' の類似度: 0.9558
'The movie was full of fun.' と 'The movie was full of rubbish.' の類似度: 0.9475
'The movie was full of excitement.' と 'The movie was full of crap.' の類似度: 0.9541
'The movie was full of excitement.' と 'The movie was full of rubbish.' の類似度: 0.9487
'The movie was full of crap.' と 'The movie was full of rubbish.' の類似度: 0.9807


## 84. 平均による文ベクトル

以下の文の全ての組み合わせに対して、最終層の埋め込みベクトルの平均を用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."

In [9]:
texts = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

mean_vectors = []
for text in texts:
  inputs = tokenizer(text, return_tensors = 'pt', padding = True, truncation = True)
  with torch.no_grad():
    outputs = model(**inputs)
  last_hidden_state = outputs.last_hidden_state

  attention_mask = inputs['attention_mask']
  mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float() # attention_maskの形状を[1, シーケンス長] → [1, シーケンス長, 768]
  masked_vector = last_hidden_state * mask # マスクを適用して、パディング部分のベクトルを0にする
  summed = torch.sum(masked_vector, dim = 1)
  token_counts = torch.sum(attention_mask, dim = 1)
  mean_vector = summed / (token_counts.unsqueeze(-1) + 1e-9)

  mean_vectors.append(mean_vector)

print("文の組み合わせに対するコサイン類似度: ")
stacked_vectors = torch.cat(mean_vectors, dim = 0) # stacked_vectors の形状は [4, 768] になる
similarity_matrix = cosine_similarity(stacked_vectors.cpu().numpy()) # NumPy配列に変換して渡す
for i in range(len(texts)):
  for j in range(i + 1, len(texts)):
    print(f"'{texts[i]}' と '{texts[j]}' の類似度: {similarity_matrix[i][j]:.4f}")

文の組み合わせに対するコサイン類似度: 
'The movie was full of fun.' と 'The movie was full of excitement.' の類似度: 0.9568
'The movie was full of fun.' と 'The movie was full of crap.' の類似度: 0.8490
'The movie was full of fun.' と 'The movie was full of rubbish.' の類似度: 0.8169
'The movie was full of excitement.' と 'The movie was full of crap.' の類似度: 0.8352
'The movie was full of excitement.' と 'The movie was full of rubbish.' の類似度: 0.7938
'The movie was full of crap.' と 'The movie was full of rubbish.' の類似度: 0.9226


## 85. データセットの準備

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) から訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、さらに全てのテキストはトークン列に変換せよ。

In [6]:
def load_data(file_path: str) -> tuple[list, list]:
  data = pd.read_csv(file_path, sep = '\t')
  return data['sentence'].tolist(), data['label'].tolist()

def tokenize_texts(texts: list) -> list:
  tokenized_texts = []
  for text in texts:
    tokens = tokenizer(text, return_tensors = 'pt')['input_ids']
    tokenized_texts.append(tokens)
  return tokenized_texts

In [7]:
train_sentence, train_label = load_data('/content/drive/MyDrive/nlp100/lesson09/SST-2/train.tsv')
dev_sentence, dev_label = load_data('/content/drive/MyDrive/nlp100/lesson09/SST-2/dev.tsv')
train_tokenized_sentence = tokenize_texts(train_sentence)
dev_tokenized_sentence = tokenize_texts(dev_sentence)

In [8]:
train_data = pd.DataFrame({'sentence': train_tokenized_sentence, 'label': train_label})
train_data

,sentence,label
0,"[[tensor(101), tensor(5342), tensor(2047), ten...",0
1,"[[tensor(101), tensor(3397), tensor(2053), ten...",0
2,"[[tensor(101), tensor(2008), tensor(7459), ten...",1
3,"[[tensor(101), tensor(3464), tensor(12580), te...",0
4,"[[tensor(101), tensor(2006), tensor(1996), ten...",0
...,...,...
67344,"[[tensor(101), tensor(1037), tensor(26380), te...",1
67345,"[[tensor(101), tensor(21782), tensor(1010), te...",0
67346,"[[tensor(101), tensor(2012), tensor(10910), te...",1
67347,"[[tensor(101), tensor(1037), tensor(5776), ten...",1


## 86. ミニバッチの作成

85で読み込んだ訓練データの一部（例えば冒頭の4事例）に対して、パディングなどの処理を行い、トークン列の長さを揃えてミニバッチを構成せよ。

In [9]:
sample_sentences = train_sentence[:4]
sample_labels = train_label[:4]
sample_tokenized_sentences = train_tokenized_sentence[:4]
encoded = tokenizer(sample_sentences, return_tensors = 'pt', padding = True, truncation = True)

In [14]:
print("元のテキスト:")
for text in sample_sentences:
    print(f"- {text}")

print("\nトークン列:")
for tokens in sample_tokenized_sentences:
    print(f"- {tokens}")

print("\nパディング後のトークンID:")
print(encoded["input_ids"])

print("\nアテンションマスク:")
print(encoded["attention_mask"])

print("\nラベル:")
print(torch.tensor(sample_labels))

元のテキスト:
- hide new secretions from the parental units 
- contains no wit , only labored gags 
- that loves its characters and communicates something rather beautiful about human nature 
- remains utterly satisfied to remain the same throughout 

トークン列:
- tensor([[  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102]])
- tensor([[  101,  3397,  2053, 15966,  1010,  2069,  4450,  2098, 18201,  2015,
           102]])
- tensor([[  101,  2008,  7459,  2049,  3494,  1998, 10639,  2015,  2242,  2738,
          3376,  2055,  2529,  3267,   102]])
- tensor([[  101,  3464, 12580,  8510,  2000,  3961,  1996,  2168,  2802,   102]])

パディング後のトークンID:
tensor([[  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102,
             0,     0,     0,     0,     0],
        [  101,  3397,  2053, 15966,  1010,  2069,  4450,  2098, 18201,  2015,
           102,     0,     0,     0,     0],
        [  101,  2008,  7459,  2049,  3494,  1998, 10639,  2015,  2242,  2738,
          33

## 87. ファインチューニング

訓練セットを用い、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

In [10]:
train_tokenized_sentence = tokenizer(train_sentence, return_tensors = 'pt', padding = True, truncation = True)
dev_tokenized_sentence = tokenizer(dev_sentence, return_tensors = 'pt', padding = True, truncation = True)
train_tokenized_sentence['labels'] = train_label
dev_tokenized_sentence['labels'] = dev_label
train_dataset = Dataset.from_dict(train_tokenized_sentence)
dev_dataset = Dataset.from_dict(dev_tokenized_sentence)

In [11]:
train_dataset

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 67349
})

In [12]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis = 1) # 最も確率の高いクラスを選択
    return accuracy_metric.compute(predictions = predictions, references = labels)

In [19]:
model = AutoModelForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels = 2)
accuracy_metric = evaluate.load("accuracy")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/nlp100/lesson09/Fine-Tuning/results',
    num_train_epochs = 3,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size = 64,
    logging_dir = '/content/drive/MyDrive/nlp100/lesson09/Fine-Tuning/logs',
    report_to = 'none'
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = dev_dataset,
    compute_metrics = compute_metrics
)

trainer.train()
trainer.save_model('/content/drive/MyDrive/nlp100/lesson09/Fine-Tuning/results/final_model')
eval_results = trainer.evaluate()

print(f"検証セットの正解率: {eval_results['eval_accuracy']:.4f}")

Step,Training Loss
500,0.246900
1000,0.171100
1500,0.102500
2000,0.096700


## 88. 極性分析

問題87でファインチューニングされたモデルを用いて、以下の文の極性を予測せよ。

- "The movie was full of incomprehensibilities."
- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


In [18]:
model_path = '/content/drive/MyDrive/nlp100/lesson09/Fine-Tuning/results/final_model'
original_model_name = 'google-bert/bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(original_model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

TypeError: stat: path should be string, bytes, os.PathLike or integer, not NoneType

In [ ]:
texts = [
    "The movie was full of incomprehensibilities.",
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

inputs = tokenizer(texts, return_tensors = 'pt', padding = True, truncation = True)
with torch.no_grad():
    outputs = model(**inputs)

# モデルの出力（logits）から確率を計算
logits = outputs.logits
probabilities = torch.softmax(logits, dim = -1)
predicted_ids = torch.argmax(probabilities, dim = -1)
id2label = {0: "ネガティブ", 1: "ポジティブ"}

# 結果を表示
for i, text in enumerate(texts):
    pred_id = predicted_ids[i].item()
    pred_label = id2label[pred_id]
    confidence = probabilities[i][pred_id].item()
    print("-" * 30)
    print(f"テキスト: 「{text}」")
    print(f"予測結果: {pred_label} (信頼度: {confidence:.4f})")

## 89. アーキテクチャの変更

問題87とは異なるアーキテクチャ（例えば[CLS]トークンを用いるか、各トークンの最大値プーリングを用いるなど）の分類モデルを設計し、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。